# MiniCells — Kaggle continuation engineering acceptance

**CPU only. No R3 retraining or R3b experiment.** Keep Accelerator set to **None**.

- **A:** defaults → Save & Run All → retain `minicells-engineering-run` at Step 6, PAUSED.
- **B:** Add Input → A's exact Notebook Output. Set `RESUME_INPUT` to its mounted run directory, `REQUIRE_RESUME=True`, `STOP_AFTER_STEP=10`. Run All. Verify `RESTORED_FROM_STEP=6`, `STARTED_FROM_RANDOM_INIT=false`.
- **C:** Add B's exact Output. Set `ACTION='evaluate'`, its `RESUME_INPUT`, and `EVALUATE_STEP=6`. Run All. Retain historical evaluation and uninterrupted-control comparison.

Use a separate engineering Notebook so an active R3 session remains untouched. If self-history inputs are unsupported, use two alternating engineering Notebooks with this same template. Send A/B/C version links and output metadata for verification. No additional tests/Pilot are run on Kaggle.

The code SHA below is frozen. Do not change it between A/B/C. Working files are not durable until the version successfully saves and can be mounted by the next Session.


In [ ]:
from pathlib import Path

# CPU engineering acceptance only. Keep GPU accelerator set to None.
ACTION = "train"  # train | evaluate | verify
RESUME_INPUT = None  # mounted previous Output directory containing continuation.json
STOP_AFTER_STEP = 6  # A: 6; B: 10
REQUIRE_RESUME = False  # B: True
EVALUATE_ARM = "QC-R3"
EVALUATE_STEP = 6
PUBLISH_SUMMARY = False  # optional; uses existing Kaggle Secret GITHUB_TOKEN


In [ ]:
import os
import re
import subprocess
import sys
import json

SOURCE_SHA = "cb01879fb5a82c162acc280d59e48301e26385d4"  # frozen code; never resolve a movable branch
CODE = Path("/kaggle/working/minicells-continuation-code-" + SOURCE_SHA[:12])
OUTPUT = Path("/kaggle/working/minicells-engineering-run")
EVALUATION = Path("/kaggle/working/minicells-engineering-evaluation")
ENTRY = CODE / "research/projects/cellular-language-models/engineering/kaggle-continuation/run.py"
os.environ.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")
if not re.fullmatch("[0-9a-f]{40}", SOURCE_SHA):
    raise ValueError("SOURCE_SHA must be the frozen full commit")
if ACTION not in ("train", "evaluate", "verify"):
    raise ValueError("unknown engineering action")
if (REQUIRE_RESUME or ACTION != "train") and not RESUME_INPUT:
    raise ValueError("RESUME_INPUT_MISSING: attach the exact previous Notebook Output first")
if RESUME_INPUT and not str(RESUME_INPUT).startswith("/kaggle/input/"):
    raise ValueError("Use the real read-only /kaggle/input mount for cross-session acceptance")
if RESUME_INPUT and ACTION == "train" and not REQUIRE_RESUME:
    raise ValueError("Set REQUIRE_RESUME=True when resuming; input cannot silently start a new run")

def execute(args, cwd=None):
    print("[Continuation] " + " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True)

# No unit tests, dataset downloads, GPU Pilot or dependency reinstall in Kaggle.
if not (CODE / ".git").exists():
    CODE.mkdir(parents=True, exist_ok=False)
    execute(["git", "init", "--quiet", CODE])
    execute(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], CODE)
    execute(["git", "sparse-checkout", "init", "--no-cone"], CODE)
    execute(["git", "sparse-checkout", "set", "--no-cone", "/pyproject.toml", "/src/", "/shared/python/",
             "/research/projects/cellular-language-models/engineering/kaggle-continuation/",
             "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
             "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
             "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
             "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
             "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/"], CODE)
    execute(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA], CODE)
    execute(["git", "checkout", "--detach", "FETCH_HEAD"], CODE)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CODE, text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError("SOURCE_SHA_MISMATCH")
print({"scope": "SYNTHETIC_CPU_ENGINEERING_ONLY", "source_sha": actual,
       "action": ACTION, "formal_training_authorized": False}, flush=True)


In [ ]:
base = [sys.executable, ENTRY, "--source-sha", SOURCE_SHA]
if ACTION == "train":
    args = base + ["--mode", "auto", "--output", OUTPUT, "--stop-after-step", str(STOP_AFTER_STEP),
                   "--require-resume", str(REQUIRE_RESUME).lower()]
    if RESUME_INPUT:
        args += ["--checkpoint-input", RESUME_INPUT]
    execute(args, CODE)
    print("Save & Run All must successfully retain Output:", OUTPUT, flush=True)
elif ACTION == "verify":
    execute(base + ["--mode", "verify-checkpoint", "--checkpoint-input", RESUME_INPUT], CODE)
else:
    execute(base + ["--mode", "evaluate-checkpoint", "--checkpoint-input", RESUME_INPUT,
                    "--arm", EVALUATE_ARM, "--step", str(EVALUATE_STEP), "--output", EVALUATION], CODE)
    # Separate tiny CPU control. Never updates the historical evaluated model.
    execute(base + ["--mode", "compare-control", "--checkpoint-input", RESUME_INPUT,
                    "--output", str(EVALUATION) + "-control"], CODE)
    print("Retain evaluation + exact-control evidence:", EVALUATION, flush=True)

if PUBLISH_SUMMARY:
    try:
        if not os.environ.get("GITHUB_TOKEN", "").strip():
            from kaggle_secrets import UserSecretsClient
            os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
        execute(base + ["--mode", "publish-only", "--checkpoint-input", OUTPUT if ACTION == "train" else RESUME_INPUT], CODE)
    except Exception as error:
        print("Metadata publication did not complete (" + type(error).__name__ + "). Saved Output remains valid; retry publish-only without training.", flush=True)
print("KAGGLE_CROSS_SESSION_RESUME=NOT_VERIFIED until actual A/B/C saved-version provenance is reviewed.", flush=True)
